# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re

def agent(query: str):
    try:
        # doing the .lower() inside the try now, so if someone accidentally
        # passes a non-string (like a number or None) it doesn't crash the
        # whole program, it just falls into the except block below
        query_lower = query.lower()

        # Route 1: math stuff -> calculator tool
        if "calculate" in query_lower:
            # pulling out just the math part, ditching the word "calculate"
            expression = query_lower.replace("calculate", "").strip()
            # keep only digits, spaces and basic math symbols so eval() doesn't choke
            expression = re.sub(r"[^0-9\.\+\-\*\/\(\)\s]", "", expression)
            expression = expression.strip()

            if expression == "":
                return {
                    "type": "error",
                    "result": "No valid math expression found in query"
                }

            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                return {
                    "type": "error",
                    "result": calc_result
                }

            return {
                "type": "calculation",
                "result": calc_result
            }

        # Route 2: keyword extraction
        elif "keywords" in query_lower:
            # if the query has "from", grab everything after that as the actual text
            text_to_process = query
            if "from" in query_lower:
                idx = query_lower.find("from")
                text_to_process = query[idx + len("from"):].strip()

            keywords = extract_keywords(text_to_process)

            if not keywords:
                return {
                    "type": "error",
                    "result": "No keywords could be extracted"
                }

            return {
                "type": "keywords",
                "result": keywords
            }

        # Route 3: anything else -> general fallback response
        else:
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. No specific tool matched, so this is a general response."
            }

    # catch-all so the agent never crashes on weird/bad input
    except Exception as e:
        return {
            "type": "error",
            "result": f"Something went wrong while processing the query: {str(e)}"
        }

### 📝 Quick explanation of what I did here

So basically the agent function checks the lowercase version of the query for keywords in the string itself (like "calculate" or "keywords") and routes it to the right tool based on that.

- For the calculate route, I strip out the word "calculate" and use a regex to keep only numbers and math symbols (+, -, *, /, parentheses) so that `eval()` in the calculator tool doesn't break on random text. If nothing valid is left after cleaning, it returns an error type instead of crashing.
- For the keywords route, if the query has the word "from" in it (like "extract keywords from ..."), I just grab everything after "from" and send that to `extract_keywords()`. If no keywords come back, it returns an error type too.
- Anything that doesn't match either condition just falls into the general/else branch and gets a plain fallback response, so the agent never returns "unknown" like the starter code did.
- The whole thing is wrapped in a try/except block so if literally anything unexpected happens (bad input, weird formatting, etc.) it still returns a proper JSON-style dict with `"type": "error"` instead of throwing an exception and killing the program.

This way every possible input always ends up producing a dictionary with both `type` and `result` keys, matching the expected output format from the assignment.

One small fix I added: I moved `query_lower = query.lower()` inside the `try` block instead of before it. Originally if `query` wasn't a string (say someone passed `123` or `None` by mistake), `.lower()` would crash immediately, before the try/except even got a chance to catch it. Now that line runs inside the try, so any weird input type also gets caught by the `except` and returned as a clean `{"type": "error", ...}` dict instead of crashing the whole notebook.

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'transforming', 'industries', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. No specific tool matched, so this is a general response."}
--------------------------------------------------


In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))